In [1]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

class AtlasEffB0(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = efficientnet_b0(weights="DEFAULT")

        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280, 128)
        )
        self.head = nn.Sequential(
            nn.Linear(135, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )

    def forward(self, images, states):
        features = self.backbone(images)
        x = torch.cat([features, states], dim=1)
        return self.head(x)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = AtlasEffB0().to(device)

In [2]:
from pathlib import Path
import json
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms

class AtlasDataset(Dataset):
    def __init__(self, root):
        self.samples = []
        self.transform = transforms.Compose([
            transforms.Resize((360, 640)),
            transforms.ToTensor(),
        ])
        root = Path(root)
        for run_dir in root.glob("run_*"):
            labels_file = run_dir / "labels.jsonl"

            with open(labels_file) as f:
                for line in f:
                    label = json.loads(line)
                    image_path = (
                        run_dir
                        / "frames"
                        / f"{label['frame_id']}.png"
                    )
                    self.samples.append((image_path, label))
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label = self.samples[idx]
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)
        state = torch.tensor([
            label["dest_x"] - label["pos_x"],
            label["dest_y"] - label["pos_y"],
            label["pos_x"],
            label["pos_y"],
            label["fwd_x"],
            label["fwd_y"],
            label["fwd_z"]
        ], dtype=torch.float32)
        output = torch.tensor([
            label["mv_fwd"],
            label["mv_right"],
            label["steer"]
        ], dtype=torch.float32)

        return image, state, output

In [3]:
from torch.utils.data import DataLoader

dataset = AtlasDataset("../dataset/godot")

loader = DataLoader(
    dataset,
    batch_size=10,
    shuffle=True,
    num_workers=0
)

In [4]:
from tqdm import tqdm

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

state_dict = torch.load(
    "weights/AtlasEffB0_v0.pt",
    weights_only=True,
    map_location=device
)
model.load_state_dict(state_dict["model_state_dict"])
optimizer.load_state_dict(state_dict["optimizer_state_dict"])
start_epoch = state_dict["epoch"] + 1

num_epochs = 2

total_epochs = start_epoch + num_epochs
model.train()
for epoch in range(start_epoch, total_epochs):
    pbar = tqdm(
        loader, 
        desc=f"Epoch {epoch + 1}/{total_epochs}",
        unit="batch"
    )

    for images, states, labels in pbar:
        images = images.to(device)
        states = states.to(device)
        labels = labels.to(device)

        prediction = model(images, states)
        loss = criterion(prediction, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, "weights/AtlasEffB0_v0.pt")

Epoch 7/7: 100%|█████| 2093/2093 [14:09<00:00,  2.46batch/s, loss=0.0078]


In [8]:
from PIL import Image
from torchvision import transforms
import json

model = AtlasEffB0().to(device)
state_dict = torch.load(
    "weights/AtlasEffB0_v0.pt",
    weights_only=True,
    map_location=device
)
model.load_state_dict(state_dict["model_state_dict"])
model.eval()

transform = transforms.Compose([
    transforms.Resize((360, 640)),
    transforms.ToTensor(),
])

run_dir = Path("../dataset/godot/run_0040")
frame_id = 194

image = Image.open(run_dir / "frames" / f"{frame_id}.png").convert("RGB")
image = transform(image)
image = image.unsqueeze(0).to(device)
label = {}

with open(run_dir / "labels.jsonl") as f:
    for line in f:
        label = json.loads(line)
        if label['frame_id'] == frame_id:
            break

state = torch.tensor(
    [[
        label["dest_x"] - label["pos_x"],
        label["dest_y"] - label["pos_y"],
        label['pos_x'],
        label['pos_y'],
        label['fwd_x'],
        label['fwd_y'],
        label['fwd_z']
    ]],
    dtype=torch.float32,
    device=device,
)
with torch.no_grad():
    prediction = model(image, state)

mv_fwd = prediction[0, 0].item()
mv_right = prediction[0, 1].item()
steer = prediction[0, 2].item()

print(f"mv_fwd  : {mv_fwd:.3f}")
print(f"mv_right: {mv_right:.3f}")
print(f"steer   : {steer:.3f}")

mv_fwd  : 1.052
mv_right: 0.023
steer   : -0.051
